# 06. 제출 추론, 패키징 및 테스트

외부에서 제공된 unlabeled `test.jsonl`을 검증하고 승격된 artifact로 추론·최종화를
수행한다. 실제 제출 패키징과 L40S smoke는 필요한 lock/license/L40S가 모두 준비된
경우에만 실행한다. 일반 Colab GPU 결과를 공식 L40S 증거로 간주하지 않는다.


## 1. 독립 Colab 부트스트랩


In [ ]:
from __future__ import annotations

import importlib.util
import json
import os
import re
import shlex
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google.colab") is not None
REPO_URL = "https://github.com/Chika1472/Writing-validation.git"
# 이 노트북 변환 시점의 소스 commit. 새 코드를 사용할 때만 의도적으로 바꾼다.
REPO_REF = os.environ.get(
    "WRITING_VALIDATION_REPO_REF",
    "30c70cb628208e004515144cc999e81d794bbe95",
).strip()
COLAB_PROJECT_DIR = Path("/content/Writing-validation")
CLONE_IF_MISSING = True

# 긴 학습은 두 값을 True로 두어 세션이 끊겨도 동일 절대경로를 유지한다.
USE_GOOGLE_DRIVE = False
DRIVE_ARTIFACT_DIR = Path("/content/drive/MyDrive/Writing-validation/artifacts")
DRIVE_HF_HOME = Path("/content/drive/MyDrive/Writing-validation/huggingface")


def run_process(command, *, cwd=None, env=None):
    command = [str(part) for part in command]
    print("$", shlex.join(command))
    return subprocess.run(
        command,
        cwd=str(cwd) if cwd else None,
        env=env,
        check=True,
    )


if IN_COLAB:
    if USE_GOOGLE_DRIVE:
        from google.colab import drive

        drive.mount("/content/drive")
    if not (COLAB_PROJECT_DIR / ".git").exists():
        if not CLONE_IF_MISSING:
            raise FileNotFoundError(f"저장소가 없습니다: {COLAB_PROJECT_DIR}")
        run_process(["git", "clone", REPO_URL, COLAB_PROJECT_DIR])
        run_process(["git", "checkout", "--detach", REPO_REF], cwd=COLAB_PROJECT_DIR)
    PROJECT_ROOT = COLAB_PROJECT_DIR.resolve()
else:
    candidates = [Path.cwd(), Path.cwd().parent]
    PROJECT_ROOT = next(
        (candidate.resolve() for candidate in candidates if (candidate / "pyproject.toml").is_file()),
        None,
    )
    if PROJECT_ROOT is None:
        raise FileNotFoundError("pyproject.toml이 있는 저장소 루트에서 실행하세요.")

os.chdir(PROJECT_ROOT)

if USE_GOOGLE_DRIVE:
    DRIVE_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
    artifact_link = PROJECT_ROOT / "artifacts"
    if artifact_link.exists() and not artifact_link.is_symlink():
        raise RuntimeError(
            "artifacts가 이미 실제 디렉터리입니다. 산출물을 이동/삭제하지 말고 "
            "새 RUN_NAMESPACE를 사용해 경로 계약을 유지하세요."
        )
    if not artifact_link.exists():
        artifact_link.symlink_to(DRIVE_ARTIFACT_DIR, target_is_directory=True)
    DRIVE_HF_HOME.mkdir(parents=True, exist_ok=True)
    os.environ["HF_HOME"] = str(DRIVE_HF_HOME)

# 변환 전 commit의 옛 설정 경로도 Colab Linux에서 안전하게 호환한다.
compatibility_links = [
    (PROJECT_ROOT / "데이터셋", PROJECT_ROOT / "dataset"),
    (PROJECT_ROOT / "qwen3_14b_zero_shot", PROJECT_ROOT / "result" / "qwen3_14b_zero_shot"),
]
if IN_COLAB:
    for alias, target in compatibility_links:
        if not alias.exists() and target.exists():
            alias.symlink_to(target, target_is_directory=True)


def run_cli(label: str, enabled: bool, *arguments: object, env=None):
    command = [sys.executable, *[str(argument) for argument in arguments]]
    print(f"\n[{label}]", "RUN" if enabled else "SKIP")
    print("$", shlex.join(command))
    if not enabled:
        return None
    return subprocess.run(command, cwd=PROJECT_ROOT, env=env, check=True)


def require_paths(*paths: Path) -> None:
    missing = [str(path) for path in paths if not Path(path).exists()]
    if missing:
        raise FileNotFoundError("필수 파일이 없습니다: " + ", ".join(missing))


def require_model_revision(value: str, *, enabled: bool) -> None:
    if enabled and re.fullmatch(r"[0-9a-fA-F]{40}", value or "") is None:
        raise ValueError("MODEL_REVISION에 40자리 Hugging Face commit SHA를 입력하세요.")


def require_bf16_gpu(*, enabled: bool, recommended_vram_gb: float = 22.0) -> None:
    if not enabled:
        print("GPU 단계가 꺼져 있어 BF16 검사를 건너뜁니다.")
        return
    import torch

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA GPU가 필요합니다. Colab 런타임을 GPU로 변경하세요.")
    if not torch.cuda.is_bf16_supported():
        raise RuntimeError("이 파이프라인은 BF16이 필요합니다. T4 대신 L4/A100급을 선택하세요.")
    properties = torch.cuda.get_device_properties(0)
    vram_gb = properties.total_memory / 1024**3
    print({"gpu": properties.name, "vram_gb": round(vram_gb, 1), "bf16": True})
    if vram_gb < recommended_vram_gb:
        print(f"경고: 권장 VRAM {recommended_vram_gb:.0f}GB보다 작아 OOM 가능성이 큽니다.")


def show_json(path: Path):
    path = Path(path)
    if not path.exists():
        print("없음:", path)
        return None
    payload = json.loads(path.read_text(encoding="utf-8"))
    print(json.dumps(payload, ensure_ascii=False, indent=2)[:12000])
    return payload


commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip()
if IN_COLAB and commit != REPO_REF:
    raise RuntimeError(
        f"기존 Colab checkout {commit}이 고정 REPO_REF {REPO_REF}와 다릅니다. "
        "새 런타임/새 PROJECT_DIR를 사용하거나 모든 노트북에서 같은 ref를 지정하세요."
    )
print({
    "in_colab": IN_COLAB,
    "project_root": str(PROJECT_ROOT),
    "git_commit": commit,
    "artifacts": str((PROJECT_ROOT / "artifacts").resolve()),
    "hf_home": os.environ.get("HF_HOME"),
})


In [ ]:
INSTALL_DEPENDENCIES = IN_COLAB
PROJECT_EXTRAS = "qwen,test"

if INSTALL_DEPENDENCIES:
    run_process(
        [sys.executable, "-m", "pip", "install", "-q", "-e", f".[{PROJECT_EXTRAS}]"],
        cwd=PROJECT_ROOT,
    )
else:
    print("로컬 검증에서는 기존 환경을 사용합니다.")


## 2. 외부 입력과 실행 스위치


In [ ]:
# 실행 전에 반드시 실제 값을 채운다. moving tag나 branch 이름은 허용되지 않는다.
MODEL_REVISION = os.environ.get("MODEL_REVISION", "").strip()
ALLOW_MODEL_DOWNLOAD = False
RUN_NAMESPACE = "v1"

TEST_PATH = PROJECT_ROOT / "dataset" / "test.jsonl"
TEMPLATE_CONFIG = PROJECT_ROOT / "configs" / "inference_l40s.yaml"
# scorer artifact namespace와 runtime config namespace를 분리한다.
SCORER_RUN_NAMESPACE = RUN_NAMESPACE
RUNTIME_CONFIG_TAG = f"{RUN_NAMESPACE}_colab"  # 공식 package는 예: "v1_l40s"
INFERENCE_CONFIG = PROJECT_ROOT / "artifacts" / "runtime_configs" / f"inference_{RUNTIME_CONFIG_TAG}.yaml"
REQUIREMENTS_LOCK = PROJECT_ROOT / "requirements-lock.txt"
LICENSE_FILES = [PROJECT_ROOT / "LICENSE", PROJECT_ROOT / "THIRD_PARTY_NOTICES.md"]
HF_HOME_PATH = Path(os.environ.get("HF_HOME", "")) if os.environ.get("HF_HOME") else None

SCORER_EXPERIMENT_ID = f"qwen3_scorer_{SCORER_RUN_NAMESPACE}"
SCORER_SEED = 42
FIXED_EPOCH = 3
SCORER_NAME = f"{SCORER_EXPERIMENT_ID}_seed{SCORER_SEED}"
SCORER_CHECKPOINTS = [
    PROJECT_ROOT / "artifacts" / "models" / SCORER_EXPERIMENT_ID
    / f"seed_{SCORER_SEED}__fold_{fold}" / f"epoch_{FIXED_EPOCH}"
    for fold in range(5)
]
SCORING_MODE = "qwen_ensemble"  # 또는 "qwen_multisource_stacker"
QWEN_CALIBRATOR = PROJECT_ROOT / "artifacts" / "calibrators" / f"{SCORER_EXPERIMENT_ID}_affine.json"
BASELINE_ARTIFACT = None
ANCHOR_ARTIFACT = None
ASSESSMENT_ARTIFACT = None
STACKER_ARTIFACT = None
STACKER_SOURCE_ALIASES = {
    "qwen": "qwen",
    "baseline": None,
    "anchor": None,
    "assessment": None,
}
RATIONALE_MODE = "deterministic"  # 또는 "adapter"
RATIONALE_CHECKPOINT = None
EXPECTED_TEST_ROWS = 400
# Colab 기능 검증은 None, 공식 package/L40S 검증은 "L40S"로 새 config를 만든다.
REQUIRED_GPU_NAME_CONTAINS = None

RUN_STATIC_TESTS = True
RUN_TEST_INPUT_CHECK = False
BUILD_RUNTIME_CONFIG = False
RUN_SUBMISSION_ENGINE = False
RUN_PACKAGE = False
RUN_L40S_SMOKE = False

require_model_revision(
    MODEL_REVISION,
    enabled=BUILD_RUNTIME_CONFIG or RUN_SUBMISSION_ENGINE or RUN_PACKAGE or RUN_L40S_SMOKE,
)
require_bf16_gpu(enabled=RUN_SUBMISSION_ENGINE or RUN_L40S_SMOKE)


download_flag = ["--allow-download"] if ALLOW_MODEL_DOWNLOAD else []


## 3. 정적 회귀 테스트


In [ ]:
run_cli("full unit tests", RUN_STATIC_TESTS, "-m", "pytest", "-q")


## 4. test JSONL 계약 검사


In [ ]:
if RUN_TEST_INPUT_CHECK:
    require_paths(TEST_PATH)
    from src.data.load import load_inference_jsonl

    records = load_inference_jsonl(TEST_PATH)
    ids = [record.id for record in records]
    if len(ids) != len(set(ids)):
        raise ValueError("test id가 중복되었습니다.")
    print({"rows": len(records), "first_id": ids[0] if ids else None})
else:
    print("공식 test 파일은 저장소에 포함되어 있지 않습니다:", TEST_PATH)


## 5. fail-closed runtime config 생성

소스 template를 직접 수정하지 않고, 승격된 checkpoint/revision/source를 주입한 새 YAML을
`artifacts/runtime_configs/`에 create-only로 만든다. multisource 모드에서는 raw Qwen을
사용하므로 `QWEN_CALIBRATOR=None`이어야 하며, artifact와 alias를 쌍으로 채운다.
A100/L4 Colab 기능 검증은 `REQUIRED_GPU_NAME_CONTAINS=None`, 공식 패키지는 `"L40S"`를 쓴다.


In [ ]:
if BUILD_RUNTIME_CONFIG:
    import yaml

    require_paths(TEMPLATE_CONFIG, *SCORER_CHECKPOINTS)
    if HF_HOME_PATH is None or not HF_HOME_PATH.is_dir():
        raise FileNotFoundError("offline 실행에 사용할 고정 HF_HOME을 먼저 설정하세요.")
    if INFERENCE_CONFIG.exists():
        raise FileExistsError(f"runtime config는 덮어쓰지 않습니다: {INFERENCE_CONFIG}")
    if SCORING_MODE not in {"qwen_ensemble", "qwen_multisource_stacker"}:
        raise ValueError(f"지원하지 않는 SCORING_MODE: {SCORING_MODE}")
    if RATIONALE_MODE not in {"deterministic", "adapter"}:
        raise ValueError(f"지원하지 않는 RATIONALE_MODE: {RATIONALE_MODE}")
    if SCORING_MODE == "qwen_multisource_stacker" and QWEN_CALIBRATOR is not None:
        raise ValueError("raw Qwen stacker 모드에서는 QWEN_CALIBRATOR=None이어야 합니다.")
    if RATIONALE_MODE == "adapter" and RATIONALE_CHECKPOINT is None:
        raise ValueError("adapter rationale에는 RATIONALE_CHECKPOINT가 필요합니다.")

    optional_artifacts = {
        "baseline": BASELINE_ARTIFACT,
        "anchor": ANCHOR_ARTIFACT,
        "assessment": ASSESSMENT_ARTIFACT,
    }
    if SCORING_MODE == "qwen_ensemble":
        if any(value is not None for value in optional_artifacts.values()) or STACKER_ARTIFACT is not None:
            raise ValueError("qwen_ensemble에서는 auxiliary/stacker artifact를 모두 None으로 둡니다.")
    else:
        if STACKER_ARTIFACT is None or not any(value is not None for value in optional_artifacts.values()):
            raise ValueError("multisource에는 stacker와 하나 이상의 auxiliary artifact가 필요합니다.")
        for kind, artifact in optional_artifacts.items():
            if (artifact is None) != (STACKER_SOURCE_ALIASES[kind] is None):
                raise ValueError(f"{kind} artifact와 alias를 함께 설정하세요.")

    config = yaml.safe_load(TEMPLATE_CONFIG.read_text(encoding="utf-8"))
    config["project_root"] = str(PROJECT_ROOT)
    config["runtime"].update({
        "hf_home": str(HF_HOME_PATH.resolve()),
        "package_manifest": None,
        "requirements_lock": str(REQUIREMENTS_LOCK.resolve()) if REQUIREMENTS_LOCK.exists() else None,
        "required_gpu_name_contains": REQUIRED_GPU_NAME_CONTAINS,
        "expected_rows": EXPECTED_TEST_ROWS,
    })
    config["scoring"]["mode"] = SCORING_MODE
    config["scoring"]["qwen"].update({
        "scorer_name": SCORER_NAME,
        "checkpoints": [str(path.resolve()) for path in SCORER_CHECKPOINTS],
        "model_revision": MODEL_REVISION,
        "calibrator": str(QWEN_CALIBRATOR.resolve()) if QWEN_CALIBRATOR is not None else None,
    })
    for kind, artifact in optional_artifacts.items():
        config["scoring"][kind]["artifact"] = (
            str(Path(artifact).resolve()) if artifact is not None else None
        )
    config["scoring"]["stacker"]["artifact"] = (
        str(Path(STACKER_ARTIFACT).resolve()) if STACKER_ARTIFACT is not None else None
    )
    config["scoring"]["stacker"]["source_aliases"] = dict(STACKER_SOURCE_ALIASES)
    config["rationale"]["mode"] = RATIONALE_MODE
    config["rationale"]["checkpoint"] = (
        str(Path(RATIONALE_CHECKPOINT).resolve()) if RATIONALE_CHECKPOINT is not None else None
    )

    referenced = [
        *(path for path in optional_artifacts.values() if path is not None),
        *(path for path in (QWEN_CALIBRATOR, STACKER_ARTIFACT, RATIONALE_CHECKPOINT) if path is not None),
    ]
    require_paths(*[Path(path) for path in referenced])
    INFERENCE_CONFIG.parent.mkdir(parents=True, exist_ok=True)
    INFERENCE_CONFIG.write_text(
        yaml.safe_dump(config, allow_unicode=True, sort_keys=False),
        encoding="utf-8",
    )
    print("created", INFERENCE_CONFIG)
else:
    print("runtime config:", INFERENCE_CONFIG, "ready" if INFERENCE_CONFIG.exists() else "not created")


## 6. offline 제출 엔진

`configs/inference_l40s.yaml`의 모든 `PLACEHOLDER`와 null revision을 승격된 artifact로
먼저 채운다. raw Qwen으로 학습한 stacker라면 Qwen calibrator는 null이어야 한다.


In [ ]:
FINAL_OUTPUT = PROJECT_ROOT / "artifacts" / "final" / f"{RUNTIME_CONFIG_TAG}_run_submission.jsonl"
FINAL_LEDGER = PROJECT_ROOT / "artifacts" / "final" / f"{RUNTIME_CONFIG_TAG}_run_submission_ledger.jsonl"
BARE_OUTPUT = PROJECT_ROOT / "artifacts" / "final" / f"{RUNTIME_CONFIG_TAG}_submission.jsonl"

run_cli(
    "submission engine",
    RUN_SUBMISSION_ENGINE,
    "scripts/run_submission.py",
    "--config", INFERENCE_CONFIG,
    "--input", TEST_PATH,
    "--output", FINAL_OUTPUT,
    "--ledger", FINAL_LEDGER,
    "--bare-output", BARE_OUTPUT,
)


## 7. 제출 패키징


In [ ]:
PACKAGE_DIR = PROJECT_ROOT / "artifacts" / "packages" / f"submission_{RUNTIME_CONFIG_TAG}"
if RUN_PACKAGE:
    import yaml

    require_paths(REQUIREMENTS_LOCK, *LICENSE_FILES)
    if REQUIRED_GPU_NAME_CONTAINS != "L40S":
        raise ValueError("공식 package config는 REQUIRED_GPU_NAME_CONTAINS='L40S'여야 합니다.")
    packaged_source_config = yaml.safe_load(INFERENCE_CONFIG.read_text(encoding="utf-8"))
    if packaged_source_config["runtime"]["required_gpu_name_contains"] != "L40S":
        raise ValueError("기존 runtime config가 L40S 전용이 아닙니다. 새 RUN_NAMESPACE로 다시 만드세요.")
    if HF_HOME_PATH is None or not HF_HOME_PATH.exists():
        raise FileNotFoundError("고정된 전용 HF_HOME cache가 필요합니다.")

package_arguments: list[object] = [
    "scripts/package_submission.py",
    "--config", INFERENCE_CONFIG,
    "--output-dir", PACKAGE_DIR,
    "--requirements-lock", REQUIREMENTS_LOCK,
]
for path in LICENSE_FILES:
    package_arguments.extend(["--license-file", path])
if HF_HOME_PATH is not None:
    package_arguments.extend(["--hf-home", HF_HOME_PATH])
run_cli("package submission", RUN_PACKAGE, *package_arguments)


## 8. 실제 L40S offline smoke


In [ ]:
if RUN_L40S_SMOKE:
    import torch

    gpu_name = torch.cuda.get_device_name(0)
    if "L40S" not in gpu_name:
        raise RuntimeError(f"공식 smoke는 L40S에서만 실행합니다. 현재 GPU: {gpu_name}")

run_cli(
    "L40S offline smoke",
    RUN_L40S_SMOKE,
    "scripts/smoke_test_submission.py",
    "--config", PACKAGE_DIR / "inference_l40s.yaml",
    "--input", TEST_PATH,
    "--report", PROJECT_ROOT / "artifacts" / "reports" / f"submission_{RUNTIME_CONFIG_TAG}_l40s_smoke.json",
    "--expected-count", "400",
    "--offline",
    "--strict",
    "--verify-determinism",
    "--require-package-manifest",
)


## 9. 완료 조건

- strict parse 100%, ID/순서 보존, score byte 동일성
- package manifest와 모든 payload hash 일치
- 정확히 고정된 dependency lock 및 license/notice 포함
- 네트워크 차단 L40S 400건 시간·메모리·결정론 보고서 생성
